In [ ]:
pip install pyspark

In [1]:
import pandas as pd 
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length
from pyspark.sql.functions import explode, avg
import re

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

/opt/conda/lib/python3.7/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/09/26 20:07:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
# Set the option to display all columns
pd.set_option('display.max_columns', None)
#path = "/Users/acgid/Downloads/nba_stats/"
#Individual Statistics
file_1 = "Player Award Shares.csv" #Truth data and dependent variable
file_2 = "Player Totals.csv"
file_3 = "Player Per Game.csv"
file_4 = "Player Play By Play.csv"
file_5 = "Player Shooting.csv"
#Advanced Individual Statistics
file_6 = "Advanced.csv"
file_7 = "Per 36 Minutes.csv"
file_8 = "Per 100 Poss.csv"
#Teams Info & Stats
file_9 = "Team Totals.csv"
file_10 = "Team Stats Per Game.csv"
file_11 = "Team Stats Per 100 Poss.csv"
file_12 = "Opponent Stats Per Game.csv"
file_13 = "Opponent Stats Per 100 Poss.csv"

award_df = spark.read.csv(file_1, header=True, inferSchema=True)
plyr_tot_df = spark.read.csv(file_2, header=True, inferSchema=True)
ppg_df = spark.read.csv(file_3, header=True, inferSchema=True)
plyr_pbp_df = spark.read.csv(file_4, header=True, inferSchema=True)
plyr_shtg_df = spark.read.csv(file_5, header=True, inferSchema=True)
adv_df = spark.read.csv(file_6, header=True, inferSchema=True)
plyr_p36min_df = spark.read.csv(file_7, header=True, inferSchema=True)
plyr_p100ps_df = spark.read.csv(file_8, header=True, inferSchema=True)
tm_tot_df = spark.read.csv(file_9, header=True, inferSchema=True)
tm_pg_df = spark.read.csv(file_10, header=True, inferSchema=True)
tm_p100ps_df = spark.read.csv(file_11, header=True, inferSchema=True)
op_pg_df = spark.read.csv(file_12, header=True, inferSchema=True)
op_p100ps_df = spark.read.csv(file_13, header=True, inferSchema=True)

In [8]:
#Reference point data set that all others will join
award_df.createOrReplaceTempView("awards")
awardDF = spark.sql("SELECT *\
                FROM awards\
                WHERE season >= 1980\
                AND award = 'nba mvp'")
#awardDF.show()

#Creating SQL tables - Player tables
plyr_tot_df.createOrReplaceTempView("player_totals")
ppg_df.createOrReplaceTempView("player_pg")
plyr_pbp_df.createOrReplaceTempView("player_pbp")
plyr_shtg_df.createOrReplaceTempView("player_shooting")
adv_df.createOrReplaceTempView("advanced")
plyr_p36min_df.createOrReplaceTempView("player_p36m")
plyr_p100ps_df.createOrReplaceTempView("player_p100ps")
#Team Tables
tm_tot_df.createOrReplaceTempView("team_totals")
tm_pg_df.createOrReplaceTempView("team_pg")
tm_p100ps_df.createOrReplaceTempView("team_p100ps")
#Opponent Tables
op_pg_df.createOrReplaceTempView("opponents_pg")
op_p100ps_df.createOrReplaceTempView("opponents_p100ps")
'''
plyr_totDF = spark.sql("SELECT *\
                FROM player_totals\
                WHERE season >= 1980")
ppgDF = spark.sql("SELECT *\
                FROM player_pg\
                WHERE season >= 1980")
#ppgDF.show()
playerDF = spark.sql("SELECT *\
                FROM awards\
                INNER JOIN player_totals\
                ON awards.player_id = player_totals.player_id AND awards.age = player_totals.age\
                INNER JOIN player_pg\
                ON awards.player_id = player_pg.player_id AND awards.age = player_pg.age\
                INNER JOIN player_pbp\
                ON awards.player_id = player_pg.player_id AND awards.age = player_pg.age")
playerDF.show()
'''

'\nplyr_totDF = spark.sql("SELECT *                FROM player_totals                WHERE season >= 1980")\nppgDF = spark.sql("SELECT *                FROM player_pg                WHERE season >= 1980")\n#ppgDF.show()\nplayerDF = spark.sql("SELECT *                FROM awards                INNER JOIN player_totals                ON awards.player_id = player_totals.player_id AND awards.age = player_totals.age                INNER JOIN player_pg                ON awards.player_id = player_pg.player_id AND awards.age = player_pg.age                INNER JOIN player_pbp                ON awards.player_id = player_pg.player_id AND awards.age = player_pg.age")\nplayerDF.show()\n'

In [9]:
#Joining player tables not using SQL
award_df.filter(col("season") >= '1980')
temp1 = award_df.join(plyr_tot_df, ["player_id", "age", "player", "season"])
temp2 = temp1.join(ppg_df, ["player_id", "age", "player", "season", "team", "pos", "lg", "g", "gs"])
temp3 = temp2.join(plyr_pbp_df, ["player_id", "age", "player", "season", "team", "pos", "lg", "g", "gs", "mp"])
temp4 = temp3.join(plyr_shtg_df, ["player_id", "age", "player", "season", "team", "pos", "lg", "g", "gs", "mp"])
temp5 = temp4.join(adv_df, ["player_id", "age", "player", "season", "team", "pos", "lg", "g", "gs", "mp"])
temp6 = temp5.join(plyr_p36min_df, ["player_id", "age", "player", "season", "team", "pos", "lg", "g", "gs", "mp"])
player_df = temp6.join(plyr_p100ps_df, ["player_id", "age", "player", "season", "team", "pos", "lg", "g", "gs", "mp"])
#player_df.show()
player_df.count()

#Joining team tables together, is it necessary? 
#Win Shares is already an individual stat designed to calculate player contribution to team success.
tm_tot_df.filter(col("season") >= '1980')
#tm_tot_df.show()




DataFrame[season: int, lg: string, team: string, abbreviation: string, playoffs: boolean, g: string, mp: string, fg: string, fga: string, fg_percent: string, x3p: string, x3pa: string, x3p_percent: string, x2p: string, x2pa: string, x2p_percent: string, ft: string, fta: string, ft_percent: string, orb: string, drb: string, trb: string, ast: string, stl: string, blk: string, tov: string, pf: string, pts: string]